# Copilot Audit Log — Direct Ingester (Fabric)

End-to-end ingestion notebook that **calls Microsoft Graph directly** from inside Fabric, parses the audit-log records on the fly, and writes a Delta table the **AI-in-One Dashboard** and **AI Business Value Dashboard** PBITs consume directly.

**No PowerShell, no intermediate CSVs, no separate parser.** Replaces the previous flow:

```
OLD:  PowerShell script → local CSV → manual move into Files/audit_raw/ → Copilot_Audit_Log_Parser.ipynb → Delta
NEW:  This notebook (Graph → parse → Delta)
```

**Output**: Lakehouse Delta table `Copilot_Interactions_Parsed` (same name + schema as the existing parser, so the PBIT works without changes).

**Schedule**: weekly (matches Microsoft Graph's 7-day audit-log query window). Use the **Schedule** button at the top of the notebook.

**Permissions**: app registration with `AuditLogsQuery.Read.All` (Application permission, admin-consented). No `Sites.Selected` needed — this path doesn't touch SharePoint.


## 1. Configuration

Fill in the four config values below. The client secret should ideally come from a Key Vault — see the commented alternative.

**Lakehouse Schemas note**: the default `OUTPUT_TABLE` is `dbo.Copilot_Interactions_Parsed`. This works for both schema-enabled Lakehouses (the new Fabric default) and non-schema Lakehouses — `dbo` is the implicit default schema either way. If your Lakehouse uses a different schema, change the prefix.


In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'                  # Entra → Overview → Tenant ID
CLIENT_ID     = '<your-app-reg-client-id>'            # Entra → App registrations → your app → Application (client) ID
CLIENT_SECRET = '<your-client-secret-value>'          # The secret VALUE (not Secret ID). Prefer Key Vault — see below.

LOOKBACK_DAYS = 7                                     # Graph caps audit-log queries at 7 days back per request.
OUTPUT_TABLE  = 'dbo.Copilot_Interactions_Parsed'     # Delta table consumed by the PBIT (schema-qualified for Lakehouse Schemas mode)
WRITE_MODE    = 'overwrite'                           # 'overwrite' for full snapshots; 'append' for incremental

# --- Production: load secret from Key Vault instead of hardcoding -------------------
# from notebookutils.credentials import getSecret
# CLIENT_SECRET = getSecret('https://<your-vault>.vault.azure.net/', 'CopilotAuditAppSecret')


## 2. Authenticate to Microsoft Graph

Service principal (client secret) flow — same auth pattern as the PowerShell scripts.


In [ ]:
import requests

def get_graph_token(tenant_id: str, client_id: str, client_secret: str) -> str:
    url  = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token'
    body = {
        'client_id':     client_id,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': client_secret,
        'grant_type':    'client_credentials',
    }
    r = requests.post(url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

token   = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
print('✓ Graph token acquired.')


## 3. Create audit log query

POST `/security/auditLog/queries` with the date range. Returns a query ID; the actual records aren't ready yet — Purview processes the query asynchronously.


In [ ]:
from datetime import datetime, timezone, timedelta

end_date   = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

query_body = {
    'displayName':         f'Copilot Interactions {start_date:%Y%m%d}-{end_date:%Y%m%d}',
    'filterStartDateTime': start_date.isoformat(),
    'filterEndDateTime':   end_date.isoformat(),
    'recordTypeFilters':   ['copilotInteraction'],
    'operationFilters':    [],
}

r = requests.post(
    'https://graph.microsoft.com/beta/security/auditLog/queries',
    json=query_body, headers=headers, timeout=30,
)
r.raise_for_status()
query_id = r.json()['id']
print(f'✓ Created query: {query_id}')
print(f'  Date range: {start_date:%Y-%m-%d %H:%M} → {end_date:%Y-%m-%d %H:%M}')


## 4. Poll until query succeeds

Purview processes audit-log queries asynchronously. Typical processing time is 5–15 minutes for small tenants, up to 30 minutes for large ones. The cell below polls every 30 seconds for up to 60 attempts (30 minutes total).


In [ ]:
import time

POLL_INTERVAL_SEC = 30
MAX_POLLS         = 60

for i in range(1, MAX_POLLS + 1):
    r = requests.get(
        f'https://graph.microsoft.com/beta/security/auditLog/queries/{query_id}',
        headers=headers, timeout=30,
    )
    r.raise_for_status()
    status = r.json()['status']
    if status == 'succeeded':
        print(f'✓ Query succeeded after {i} poll(s).')
        break
    if status in ('failed', 'cancelled'):
        raise RuntimeError(f'Query failed with status: {status}')
    print(f'  [{i}/{MAX_POLLS}] Status: {status}. Sleeping {POLL_INTERVAL_SEC}s...')
    time.sleep(POLL_INTERVAL_SEC)
else:
    raise TimeoutError(f'Query did not succeed within {MAX_POLLS * POLL_INTERVAL_SEC}s')


## 5. Fetch records (paginated)

Pages through `/records?$top=999` following `@odata.nextLink` until exhausted. For very large tenants (millions of records) this can take a few minutes.


In [ ]:
records = []
next_url = f'https://graph.microsoft.com/beta/security/auditLog/queries/{query_id}/records?$top=999'

while next_url:
    r = requests.get(next_url, headers=headers, timeout=60)
    r.raise_for_status()
    data = r.json()
    records.extend(data.get('value', []))
    next_url = data.get('@odata.nextLink')
    print(f'  Fetched {len(records):,} records so far...')

print(f'✓ Total records: {len(records):,}')


## 6. Build a Spark DataFrame from the raw records

Each record from Graph has its `auditData` field as a structured JSON object. We capture it as a JSON string in a column called `AuditData` so the downstream parser logic (which is identical to `Copilot_Audit_Log_Parser.ipynb`) can `from_json` it back into a typed struct.


In [ ]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# Convert records to a list of dicts with AuditData re-serialised as a string,
# matching the schema the existing parser expects from raw CSV input.
raw_rows = [{
    'RecordId':                  r.get('id'),
    'CreationDate':              r.get('createdDateTime'),
    'RecordType':                r.get('auditLogRecordType'),
    'Operation':                 r.get('operation'),
    'AuditData':                 json.dumps(r.get('auditData', {})) if r.get('auditData') else None,
    'AssociatedAdminUnits':      json.dumps(r.get('associatedAdminUnits', [])),
    'AssociatedAdminUnitsNames': json.dumps(r.get('associatedAdminUnitsNames', [])),
} for r in records]

raw = spark.createDataFrame(raw_rows)
print(f'Raw DataFrame rows: {raw.count():,}')


## 7. Parse `AuditData` JSON

Schema mirrors the existing `Copilot_Audit_Log_Parser.ipynb` — only the fields the PBIT consumes.


In [ ]:
audit_schema = StructType([
    StructField('CreationTime', StringType()),
    StructField('UserId', StringType()),
    StructField('Workload', StringType()),
    StructField('ApplicationName', StringType()),
    StructField('ClientRegion', StringType()),
    StructField('AgentId', StringType()),
    StructField('AgentName', StringType()),
    StructField('AppIdentity', StructType([
        StructField('AppId', StringType()),
        StructField('DisplayName', StringType()),
        StructField('PublisherId', StringType()),
    ])),
    StructField('CopilotEventData', StructType([
        StructField('AppHost', StringType()),
        StructField('ThreadId', StringType()),
        StructField('SensitivityLabelId', StringType()),
        StructField('Contexts', ArrayType(StructType([
            StructField('Type', StringType())
        ]))),
        StructField('AISystemPlugin', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('Name', StringType())
        ]))),
        StructField('ModelTransparencyDetails', ArrayType(StructType([
            StructField('ModelName', StringType())
        ]))),
        StructField('AccessedResources', ArrayType(StructType([
            StructField('Type', StringType()),
            StructField('Action', StringType()),
            StructField('SiteUrl', StringType()),
            StructField('SensitivityLabelId', StringType()),
        ]))),
        StructField('Messages', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('isPrompt', StringType())
        ]))),
    ])),
])

parsed = (raw
    .filter(F.col('AuditData').isNotNull() & (F.length('AuditData') > 10))
    .withColumn('j', F.from_json('AuditData', audit_schema))
    .filter(F.col('j').isNotNull()))


## 8. Flatten + fan out per (prompt × resource)

Mirrors the M-query exactly: drop responses, expand resources. One row per prompt-resource pair.


In [ ]:
flat = (
    parsed
    .select(
        F.col('j.CreationTime').cast('timestamp').alias('CreationDate'),
        F.col('j.AgentId').alias('AgentId'),
        F.col('j.AgentName').alias('AgentName'),
        F.col('j.AppIdentity.AppId').alias('AppIdentity_AppId'),
        F.col('j.AppIdentity.DisplayName').alias('AppIdentity_DisplayName'),
        F.col('j.AppIdentity.PublisherId').alias('AppIdentity_PublisherId'),
        F.col('j.ApplicationName').alias('ApplicationName'),
        F.col('j.ClientRegion').alias('ClientRegion'),
        F.col('j.UserId').alias('Audit_UserId'),
        F.lower(F.trim(F.col('j.UserId'))).alias('Audit_UserId_Normalized'),
        F.col('j.Workload').alias('Workload'),
        F.col('j.CopilotEventData.AppHost').alias('AppHost'),
        F.col('j.CopilotEventData.ThreadId').alias('ThreadId'),
        F.col('j.CopilotEventData.SensitivityLabelId').alias('SensitivityLabelId'),
        F.col('j.CopilotEventData.Contexts')[0]['Type'].alias('Context_Type'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Id'].alias('AISystemPlugin_Id'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Name'].alias('AISystemPlugin_Name'),
        F.col('j.CopilotEventData.ModelTransparencyDetails')[0]['ModelName']
            .alias('ModelTransparencyDetails_ModelName'),
        F.col('j.CopilotEventData.AccessedResources').alias('Resources'),
        F.col('j.CopilotEventData.Messages').alias('Messages'),
        F.size('j.CopilotEventData.AccessedResources').alias('Resource_Count'),
    )
    .withColumn('msg', F.explode('Messages'))
    .filter(F.lower(F.col('msg.isPrompt').cast('string')) == 'true')
    .withColumn(
        'res',
        F.explode_outer(
            F.when(F.size('Resources') > 0, F.col('Resources'))
             .otherwise(F.array(F.struct(
                 F.lit(None).cast(StringType()).alias('Type'),
                 F.lit(None).cast(StringType()).alias('Action'),
                 F.lit(None).cast(StringType()).alias('SiteUrl'),
                 F.lit(None).cast(StringType()).alias('SensitivityLabelId'),
             )))
        ),
    )
    .select(
        'CreationDate', 'AgentId', 'AgentName',
        'AppIdentity_AppId', 'AppIdentity_DisplayName', 'AppIdentity_PublisherId',
        'ApplicationName', 'ClientRegion',
        'Audit_UserId', 'Audit_UserId_Normalized', 'Workload',
        'AppHost', 'ThreadId', 'SensitivityLabelId',
        'Context_Type',
        'AISystemPlugin_Id', 'AISystemPlugin_Name',
        'ModelTransparencyDetails_ModelName',
        F.col('res.Type').alias('AccessedResource_Type'),
        F.col('res.Action').alias('AccessedResource_Action'),
        F.col('res.SiteUrl').alias('AccessedResource_SiteUrl'),
        F.col('res.SensitivityLabelId').alias('AccessedResource_SensitivityLabelId'),
        F.col('msg.Id').alias('Message_Id'),
        F.col('msg.isPrompt').alias('Message_isPrompt'),
        F.coalesce(F.col('Resource_Count'), F.lit(1)).alias('Resource_Count'),
        F.to_date('CreationDate').alias('InteractionDate'),
        F.expr("date_trunc('week', CreationDate)").cast('date').alias('WeekStart'),
        F.expr("date_trunc('month', CreationDate)").cast('date').alias('MonthStart'),
    )
)


## 9. Derive `Agent_TitleID`

Same logic as the existing parser + M-query.


In [ ]:
flat = flat.withColumn('Agent_TitleID', F.expr(r'''
    CASE
      WHEN AgentId IS NULL THEN NULL
      WHEN AgentId LIKE '%CopilotStudio.Declarative.%' THEN
           split(split(AgentId, 'CopilotStudio\\.Declarative\\.')[1], '\\.')[0]
      WHEN AgentId LIKE 'P\\_%' OR AgentId LIKE 'T\\_%' THEN
           split(AgentId, '\\.')[0]
      ELSE NULL
    END
'''))


## 10. Write to Lakehouse Delta table


In [ ]:
(flat.write
    .format('delta')
    .mode(WRITE_MODE)
    .option('overwriteSchema', 'true')
    .saveAsTable(OUTPUT_TABLE))

row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows written to {OUTPUT_TABLE}: {row_count:,}')


## 11. Verify

Spot-check the output. Expect populated `AISystemPlugin_Id` (e.g. `BingWebSearch`), `Workload`, `AppHost`.


In [ ]:
spark.table(OUTPUT_TABLE).select(
    'AISystemPlugin_Id', 'Workload', 'AppHost'
).groupBy('AISystemPlugin_Id', 'Workload', 'AppHost').count().orderBy(F.desc('count')).show(20, truncate=False)


---
**Connect the PBIT**: open the Fabric variant of either dashboard and supply the **Fabric SQL Endpoint** + **Lakehouse Database** parameters as before. The PBIT reads `Copilot_Interactions_Parsed` directly via the SQL endpoint — no other parameter changes needed when switching from the CSV-based parser to this direct ingester.
